# 8. From joined points to a census-area map

The spatial join gives each library a census-area identifier.

Next we:

1. count libraries by census area;
2. join those counts back to all census polygons;
3. fill areas with no matched libraries with zero;
4. create a map.

In [ ]:
import geopandas as gpd

In [ ]:
libraries = gpd.read_file(
    "../data/processed/toronto_libraries_clean.gpkg"
)

census = gpd.read_file(
    "../data/raw/toronto_census_areas.geojson"
)

libraries = libraries.to_crs(census.crs)

library_points = libraries.copy()
library_points["geometry"] = (
    library_points.geometry.representative_point()
)

libraries_with_census = gpd.sjoin(
    library_points,
    census[["ADAUID", "geometry"]],
    how="left",
    predicate="within"
)

## Count libraries by census area

After grouping:

| ADAUID | library_count |
|---|---:|
| 35200001 | 2 |
| 35200002 | 1 |
| 35200007 | 3 |

Census areas with no libraries are absent from this small summary table.

In [ ]:
library_counts = (
    libraries_with_census
    .groupby("ADAUID")
    .size()
    .reset_index(name="library_count")
)

display(library_counts.head())

## Merge counts back to all census areas

A left merge keeps every census polygon.

```text
census polygons              library counts
--------------              --------------
Area A                       Area A → 2
Area B                       Area C → 1
Area C
         ↓ left merge
Area A → 2
Area B → missing
Area C → 1
```

Here, after checking the join, the missing count for Area B can reasonably become `0`: the census area exists, but no library point matched it.

In [ ]:
census_with_libraries = census.merge(
    library_counts,
    on="ADAUID",
    how="left"
)

census_with_libraries["library_count"] = (
    census_with_libraries["library_count"]
    .fillna(0)
)

In [ ]:
ax = census_with_libraries.plot(
    column="library_count",
    legend=True,
    edgecolor="white",
    linewidth=0.2,
    figsize=(8, 8)
)

ax.set_title("Library locations by Toronto census area")
ax.set_axis_off()